In [1]:

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

save_dir = project_root / "interface" / "models" / "pretrained"
save_dir.mkdir(parents=True, exist_ok=True)

model_file_path = save_dir / "base_model.pt"

In [2]:
import numpy as np
import torch

def simulate_structured_fmri(n_rois=50, n_timepoints=200):
    """
    Generates data containing hidden 'networks' of synchronized regions.
    Useful for validating if the GNN/Solver can detect patterns amidst noise.
    """
    # Start with base white noise
    data = np.random.randn(n_rois, n_timepoints)
    
    # Inject a strong signal (Sine Wave) to create synchrony
    # Imagine this simulates the "Default Mode Network" (DMN)
    t = np.linspace(0, 4*np.pi, n_timepoints)
    common_signal = np.sin(t) * 2.0  # Strong signal amplitude
    
    # Force regions 0, 1, 2, 3, and 4 to "sing the same tune"
    cluster_1 = [0, 1, 2, 3, 4]
    for roi in cluster_1:
        # Add the common signal to the noise (Signal + Noise)
        data[roi] += common_signal
        
    # Create a second cluster (e.g., Visual Network) with a different frequency
    fast_signal = np.sin(t * 3) * 2.0
    cluster_2 = [10, 11, 12, 13]
    for roi in cluster_2:
        data[roi] += fast_signal

    # The remaining regions act as background random noise.
    
    return data


def build_geometry_from_data(data, threshold=0.6):
    """
    Constructs the graph skeleton (edge_index) based on data similarity.
    
    Args:
        data: Matrix (N_ROIs x Timepoints), generated by the simulator.
        threshold: How strong the relationship must be to become an edge.
                   Since we injected a strong signal (amplitude 2.0), 
                   correlation will be high (>0.8). 0.6 is a safe bet.
    """
    # Calculate Pearson Correlation (Who behaves like whom?)
    # Returns a matrix (N_ROIs x N_ROIs) with values between -1 and 1
    correlation_matrix = np.corrcoef(data)
    
    # Zero out the diagonal (Self-connection) to avoid edges (0,0), (1,1)...
    # (Optional: Some GNNs like self-loops, but we can add them later via transforms)
    np.fill_diagonal(correlation_matrix, 0)
    
    # Find where the connection is strong (Absolute Value > Threshold)
    # np.where returns two arrays: source indices and target indices
    rows, cols = np.where(np.abs(correlation_matrix) > threshold)
    
    # Convert to PyTorch Geometric format [2, Num_Edges]
    # The first row contains source nodes, the second contains target nodes
    edge_index = torch.tensor([rows, cols], dtype=torch.long)
    
    return edge_index

In [3]:
import torch
import numpy as np
from interface.core.engine import QyraEngine
from interface.modeling_logic.translators import AffinityTranslator


# --- SETTINGS ---
NUM_NODES = 50
NUM_FEATURES = 200

# Instantiate the Engine (This loads weights if they exist)
engine = QyraEngine(num_features=NUM_FEATURES, model_path=str(model_file_path))

print("--- Starting Hybrid Resolution Cycle ---")

# Simulating real fMRI data
# 50 brain regions, monitored over 200 time points
brain_signals = simulate_structured_fmri(50, 200)

x = torch.tensor(brain_signals, dtype=torch.float)

# Initializes the Translator
# threshold=0.5: Ignores weak correlations
# regularization=0.5: Charges a fee of 0.5 energy to connect a node
translator = AffinityTranslator(threshold=0.5, regularization=0.5)

# Translate to Matrix Q
print("Translating brain signals to QUBO...")
q_matrix = translator.from_time_series(brain_signals)

print(f"\nMatrix ready to send to solver! Size: {len(q_matrix)}x{len(q_matrix)}")

edge_index = build_geometry_from_data(brain_signals, threshold=0.6)
print(f"Graph generated with {edge_index.shape[1]} connections.")

# We call .solve() with learn=True
solution, energy = engine.solve(x, edge_index, q_matrix, learn=True)

# Results
print(f"\n>>> Problem solved!")
print(f"Final Energy: {energy:.2f}")
print(f"Solution Vector (First 15 nodes): {solution[:15]}")

# Visual verification: We expect 0, 1, 2, 3, 4 and 10, 11, 12, 13 to be 1 (or consistently 0)
print("\nCluster 1 Check (Expected synchrony):", [solution[i] for i in range(5)])

# 2. Save the trained model for future use
engine.save_model()
print("Base model saved.")

/Users/guilhermealves/Documents/personal/qyra/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Qyra] Loading pre-trained model from: /Users/guilhermealves/Documents/personal/qyra/interface/models/pretrained/base_model.pt
--- Starting Hybrid Resolution Cycle ---
Translating brain signals to QUBO...

Matrix ready to send to solver! Size: 50x50
Graph generated with 30 connections.

>>> Problem solved!
Final Energy: -17.47
Solution Vector (First 15 nodes): [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0]

Cluster 1 Check (Expected synchrony): [1, 1, 1, 1, 1]
[Qyra] Model saved to /Users/guilhermealves/Documents/personal/qyra/interface/models/pretrained/base_model.pt
Base model saved.


Echo received by the solver.
/var/folders/t9/kfjfhkzx06n2nhmhz280v41c0000gn/T/ipykernel_79993/1541801282.py:58: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  edge_index = torch.tensor([rows, cols], dtype=torch.long)
Optimizing...
Optimization completed.
